# 서울 공공자전거 데이터 분석 실습

## 서울 공공자전거 대여 데이터

서울시 공공자전거 운영 담당자가 이렇게 말했습니다.

> "자전거가 언제, 어떤 때 많이 쓰이는지 알고 싶습니다."

이 말은 아직 **모호한 문제**입니다. 데이터를 직접 다루며 이 문제를 **분석할 수 있는 질문으로 바꾸고, 질문에 답하는 방법**을 익힙니다.

- **데이터** 서울 공공자전거의 1년(365일) × 24시간 대여 기록, 8,760행 14열
- **출처** UCI Machine Learning Repository, Seoul Bike Sharing Demand (https://archive.ics.uci.edu/dataset/560/seoul+bike+sharing+demand)

### 쓸 수 있는 도구

| 하고 싶은 일 | 도구 |
|:---|:---|
| 데이터 미리보기 | `head()` `tail()` `sample()` |
| 데이터 구조 확인 | `shape` `columns` `dtypes` `info()` |
| 변수 선택, 필터링 | `df[["열"]]` `df[조건]` `&` `\|` `unique()` |
| 열 추가 | `df["새 열"] = 계산` |
| 정렬 | `sort_values()` |
| 결측값, 중복값 확인 | `isna()` `duplicated()` |
| 빈도 계산 | `value_counts()` `len()` |
| 기술통계 | `describe()` `mean()` `median()` |
| 그룹별 집계 | `groupby()` `agg()` `idxmax()` `unstack()` |
| 상관계수 | `corr()` |
| 표 만들기 | `pd.DataFrame()` `round()` |
| 그래프 | `plot(kind="bar")` `plot(kind="hist")` `boxplot()` `plot(kind="line")` `plot(kind="scatter")` `set_ylim()` |

## 데이터 불러오기

아래 두 셀은 **이미 작성되어 있습니다.** 실행만 하세요. 이 파일은 온도 기호(°) 때문에 `encoding="latin1"`로 읽어야 글자가 깨지지 않습니다.

In [ ]:
from pathlib import Path
import pandas as pd

URL = "https://raw.githubusercontent.com/phyzikyh/computing/main/%EA%B0%95%EC%9D%98%EA%B5%90%EC%9E%AC/data/SeoulBikeData.csv"


def find_file(name):
    starts = [Path.cwd()]
    notebook = globals().get("__vsc_ipynb_file__")      # VS Code에서 노트북이 있는 위치
    if notebook:
        starts.insert(0, Path(notebook).parent)
    for start in starts:
        for folder in [start, *start.parents][:4]:
            for sub in ["", "data", "강의교재/data"]:
                path = folder / sub / name
                if path.exists():
                    return path
    return None


data_path = find_file("SeoulBikeData.csv")
source = data_path if data_path else URL          # 컴퓨터에 없으면 인터넷에서 내려받습니다.
bike = pd.read_csv(source, encoding="latin1")
print("불러온 곳:", source)

In [ ]:
# 원본의 긴 열 이름을 짧게 바꿉니다. (원본 파일은 바뀌지 않습니다.)
bike = bike.rename(columns={
    "Date": "date",
    "Rented Bike Count": "rented_count",
    "Hour": "hour",
    "Temperature(°C)": "temperature",
    "Humidity(%)": "humidity",
    "Wind speed (m/s)": "wind_speed",
    "Visibility (10m)": "visibility",
    "Dew point temperature(°C)": "dew_point",
    "Solar Radiation (MJ/m2)": "solar_radiation",
    "Rainfall(mm)": "rainfall",
    "Snowfall (cm)": "snowfall",
    "Seasons": "season",
    "Holiday": "holiday",
    "Functioning Day": "functioning_day",
})
print(bike.columns.tolist())

# 1. EDA 절차 따라 하기

분석 질문이 하나 주어졌습니다. 이 질문에 답하는 과정을 이론과 같은 순서로 밟습니다.

> 서울 공공자전거의 시간대별 평균 대여량은 어떻게 다르며, 어느 시간대에 대여가 가장 많은가?

### 1.1 분석 질문에 필요한 변수 정의

데이터를 열기 전에 먼저 생각합니다. ① 이 질문에 **필요한 변수(열)** ② **필요 없는 변수** ③ **집계 방법**을 쓰세요. 그리고 ④ 분석을 시작하기 전에 데이터가 이 질문에 답할 수 있는지 **점검해야 할 항목 네 가지**를 쓰세요.

**답**

여기에 답을 쓰세요.

### 1.2 변수의 종류 분류

`bike`의 열 14개를 **범주형**과 **연속형**으로 분류하세요. 숫자로 저장되어 있지만 범주처럼 다루는 열이 있다면 이유와 함께 쓰세요. 이 분석 질문(시간대별 평균 대여량)에서 `hour`와 `rented_count`는 각각 어느 쪽으로 다룹니까?

### 1.3 데이터프레임 확인

`bike`에는 무엇이 저장되어 있습니까? 파일 이름입니까, 표입니까? 자료형을 코드로 확인하고, 이 데이터에서 **관측치(행) 하나**와 **변수(열) 하나**가 각각 무엇을 나타내는지 설명하세요.

**힌트** `type()`으로 자료형을 확인합니다.

### 1.4 데이터 미리보기

데이터의 처음 5행, 마지막 3행, 무작위 표본 3행(`random_state=10`으로 고정)을 미리 보세요. 세 가지를 **모두** 확인하는 이유를 설명하세요.

### 1.5 데이터 크기와 변수 자료형

관측치(행)와 변수(열)의 수, 변수 이름, 변수별 자료형을 확인하세요. 수치형 변수 세 가지와 문자열(범주형) 변수 세 가지를 적으세요.

### 1.6 info()로 데이터 구조 확인

`info()`를 실행하고 `Non-Null Count`와 `Dtype`을 읽으세요. 비어 있는 값이 있는 열이 있습니까? `date`는 어떤 자료형이며, **이 질문에서** 날짜를 날짜형으로 바꿀 필요가 있습니까?

### 1.7 열 하나 꺼내기와 고유값

`bike`에서 열 하나(`season`)를 꺼내 **타입**을 확인하세요. 그다음 `season`, `holiday`, `functioning_day` 세 열이 각각 가진 **고유한 값**을 확인하세요. 계절은 몇 종류입니까?

### 1.8 변수 선택과 필터링

질문에 필요한 변수(열) 두 개만 선택해 새 이름으로 저장하고 앞부분을 확인하세요. 그다음 **18시 관측치**만 필터링해 몇 건인지 세세요.

### 1.9 복합 조건 필터링

**18시이고 기온이 25도 이상인** 관측치가 몇 건인지 구하세요. 조건 두 개를 함께 쓸 때 코드에서 주의할 점을 한 줄로 쓰세요.

### 1.10 새로운 열 만들기

비가 온 시간인지를 나타내는 열 `is_rain`을 `bike`에 만드세요. `rainfall`(강수량, mm)이 0보다 크면 `True`, 아니면 `False`입니다. 새 열을 만든 뒤 `True`와 `False`가 각각 몇 건인지 세고, 비가 온 시간이 전체의 몇 %인지 구하세요. 이 열은 뒤에서 비 오는 시간과 비 안 오는 시간을 비교할 때 사용합니다.

**힌트** 조건식 `bike["rainfall"] > 0`의 결과(True/False)를 그대로 새 열에 저장합니다. 비율은 `True`의 개수 ÷ 전체 행 수 × 100입니다.

**확인값** 비가 온 시간 528건 (6.0%)

### 1.11 정렬로 상위 관측치 확인

대여량이 **가장 큰** 관측치 5개를 날짜, 시간, 대여량, 계절과 함께 확인하세요. 어떤 공통점이 있습니까?

### 1.12 결측값 확인

변수별 결측값의 개수를 구하고, 데이터 전체의 합계도 구하세요.

### 1.13 중복값 확인

중복된 행이 몇 개인지 구하고, 있다면 중복된 행을 **모두** 표시하는 방법으로 확인하세요.

**힌트** `duplicated()`, 중복된 행을 모두 표시할 때는 `duplicated(keep=False)`

### 1.14 비정상 값 탐색

결측값도 중복값도 없습니다. 그런데 이 데이터를 그대로 분석에 사용해도 될까요? 대여량이 가장 **작은** 관측치부터 확인하세요. 비정상적이거나 설명이 필요한 값이 있습니까? 있다면 **몇 건인지**, **어떤 공통점이 있는지**를 코드로 확인하세요.

**힌트** `sort_values()`로 작은 값부터 보고, 대여량이 0인 행만 걸러 다른 변수와 함께 봅니다. 다른 열을 `value_counts()`로 세어 보면 단서가 나옵니다.

**확인값** 대여량이 0인 행 295건

### 1.15 데이터 전처리 방법 결정

이 관측치들을 어떻게 처리할지 정합니다. 다음 두 가지 방법으로 **평균 대여량을 계산해 비교**한 뒤, 전처리 방법을 결정하세요.

- **방법 A** 모든 관측치를 그대로 사용
- **방법 B** 문제가 되는 관측치를 제외하고 사용 (제외한 데이터를 `operating`이라는 이름으로 저장하세요)

**무엇을 제외했고 왜 그렇게 했는지** 전처리 기준을 글로 쓰세요. 이후 분석은 선택한 데이터로 합니다.

**확인값** 방법 A 평균 704.6건, 방법 B 평균 729.2건, `operating` 8,465행

### 1.16 이상값 탐지 기준

1.11에서 대여량이 매우 큰 관측치를 확인했습니다. 이 값들이 **이상값 후보**인지 IQR 기준으로 판정하세요. `operating`의 대여량으로 계산합니다.

- **위쪽 한계** Q3 + 1.5 × IQR (IQR = Q3 − Q1)

위쪽 한계값과, 한계를 **넘는 관측치의 수**를 구하세요. 초과한 관측치가 어느 시간대와 계절에 집중되어 있는지도 확인한 뒤, **이 관측치들을 제외해야 하는지** 판단하세요.

**힌트** `describe()`의 `25%`(Q1)와 `75%`(Q3), 한계를 넘는 행은 조건 필터링과 `value_counts()`

**확인값** 위쪽 한계 2389.0건, 한계를 넘는 관측치 152건

### 1.17 기술통계 요약

`operating`의 대여량을 요약하세요.

1. 개수, 평균, 표준편차, 사분위수, 최댓값을 한 번에 확인하세요.
2. 평균과 중앙값을 비교하고, 차이가 말해 주는 것을 쓰세요.
3. 계절과 공휴일 여부의 **빈도와 비율(%)**을 구하세요. 비율이 한쪽으로 치우친 변수가 있다면, 그것이 그룹 비교에 어떤 영향을 줄지 생각해 보세요.

**힌트** 비율(%)은 `value_counts()` ÷ 전체 행 수 × 100

**확인값** 평균 729.2건, 중앙값 542건

### 1.18 그룹별 집계와 분석 질문 답하기

처음 질문에 답합니다.

1. 시간대별로 **관측치 수, 평균, 중앙값**을 집계하세요. 표본 크기가 충분한지 `count`로 확인하세요.
2. 평균이 가장 큰 시간대를 **코드로** 찾고, 상위 3개와 하위 3개 시간대를 확인하세요.
3. 1.15의 전처리 결정이 결론을 바꾸는지 확인하세요(민감도 확인). `operating`과 `bike`(전체)로 같은 집계를 해 **나란히** 놓고 비교하세요. 가장 많은 시간대가 달라집니까? 값은 얼마나 달라집니까?
4. 질문에 **한 문장**으로 답하세요.

**힌트** 3번은 두 집계의 평균 열을 `pd.DataFrame({"운영일만": …, "전체": …})`로 나란히 놓습니다.

**확인값** 가장 많은 시간대 18시 (평균 1554.0건)

# 2. 탐색 결과로 분석 질문 만들기

데이터는 `operating`을 씁니다.

### 2.1 계절별 집계

1.18에서 시간대별로 한 집계를 이번에는 `season`별로 하세요. 관측치 수, 평균, 중앙값을 구하고, 평균 대여량이 가장 큰 계절과 가장 작은 계절은 무엇이며, 몇 배 차이가 납니까?

**확인값** 여름 평균 1034.1건, 겨울 평균 225.5건

### 2.2 공휴일 여부별 집계

`holiday`별로 같은 집계를 하세요. 1.17에서 확인한 것처럼 값이 한쪽으로 치우쳤을 수 있으므로 관측치 수도 함께 확인하세요. 어느 쪽 평균이 더 높습니까? 공휴일 기록은 몇 건입니까?

**확인값** 공휴일 기록 408건

### 2.3 공휴일 기록의 계절 구성

2.2의 차이를 그대로 믿어도 되는지 점검합니다. 공휴일 기록만 따로 만들어 1.17에서 한 방법으로 **계절별 건수와 비율(%)**을 구하고, `operating` 전체의 계절별 비율과 나란히 놓으세요. 공휴일이 어느 계절에 몰려 있습니까?

**힌트** 공휴일 행만 걸러 `value_counts()`, 두 비율은 `pd.DataFrame()`으로 나란히

**확인값** 공휴일 기록 중 겨울 47.1%

### 2.4 발견한 패턴과 남은 의문 정리

2.1에서 2.3까지의 결과에서 **발견한 패턴 세 가지**를 근거 수치와 함께 쓰세요. 그다음 **탐색으로는 설명되지 않은 채 남은 의문**을 한 가지 이상 쓰세요. 아직 질문 형태로 만들지는 않습니다.

**답**

여기에 답을 쓰세요.

### 2.5 탐색 결과에서 분석 질문 도출

2.4에서 남은 의문을 **분석 질문**으로 바꾸세요. 다섯 요소(대상, 측정값, 기준, 계산, 범위)를 모두 쓰고, 다음을 점검하세요.

1. 이 데이터에 필요한 변수가 있습니까?
2. 계산하면 답이 나옵니까? ("달라 보인다", "원인이다"가 아니라)
3. 비교하는 그룹의 **표본 크기가 충분**합니까?
4. 이 질문의 답이 **지금까지의 탐색 결과에 이미 나와 있지는 않습니까?** 이미 나와 있다면 새로운 분석이 필요하지 않은 질문입니다.

**질문 형식의 예** (위의 의문과 다른 주제입니다. 형식만 참고하세요.)

| 요소 | 예 |
|:---|:---|
| 대상 | 한 시간 동안의 대여 기록 |
| 측정값 | 대여량(`rented_count`) |
| 기준 | 시간대(8시와 18시) |
| 계산 | 평균 |
| 범위 | 이 자료의 1년, 운영한 시간만 |

질문 "운영한 시간 중 8시와 18시의 평균 대여량은 얼마나 다른가?"

**답**

여기에 답을 쓰세요.

### 2.6 도출한 분석 질문에 답하기

2.5에서 정의한 분석 질문에 답하는 코드를 작성하세요. 지금까지의 탐색에서 하지 않은 **새로운 비교**여야 합니다. 결과를 해석할 때 그룹의 **표본 크기**를 함께 확인하고, 2.2의 결과와 **비교**해서 질문에 한 문장으로 답하세요.

**힌트** 두 기준으로 함께 나누어 비교할 때는 `groupby(["기준1", "기준2"])`로 집계한 뒤 `unstack()`으로 표를 펼칩니다. 평균과 함께 관측치 수(`count()`)도 구합니다.

### 2.7 결과 종합과 제안

처음의 모호한 문제("자전거가 언제, 어떤 때 많이 쓰이는지 알고 싶다")로 돌아갑니다. 지금까지의 분석 결과를 종합해 답하세요.

1. 이 데이터로 **확인할 수 있는 사실(인사이트)** 세 가지를 쓰세요. 각각 근거 수치를 함께 쓰세요.
2. 이 데이터만으로는 **확인할 수 없는 것** 두 가지를 쓰세요. 왜 확인할 수 없는지도 설명하세요.
3. 운영 담당자에게 **한 가지를 제안**한다면 무엇을 제안하겠습니까? **근거 수치**와, 그 근거의 **한계**를 함께 쓰세요.
4. 제안의 근거를 강화하려면 **어떤 추가 데이터가 필요**합니까?

**답**

여기에 답을 쓰세요.

# 3. 그래프로 보여 주기

데이터를 **그래프로 그리고, 그래프를 문장으로 해석**합니다. 그래프는 분석 질문에 답하는 도구이므로, 모든 문항에서 **무엇을 알고 싶어서 어떤 그래프를 고르는지**를 먼저 생각합니다.

아래 셀은 **이미 작성되어 있습니다.** 그래프에 한글을 쓰기 위한 글꼴 설정과 제목·축 이름·막대 값을 붙이는 도우미 함수 `decorate()`, `show_values()`를 준비합니다. 실행만 하세요. Colab에서는 한글 글꼴이 기본으로 없으므로 다음 두 줄을 **먼저 실행한 뒤 런타임을 다시 시작**합니다.

```
!apt-get -qq -y install fonts-nanum
!fc-cache -fv
```

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

installed_fonts = {font.name for font in fm.fontManager.ttflist}
for font_name in ["Malgun Gothic", "AppleGothic", "NanumGothic", "Noto Sans CJK KR"]:
    if font_name in installed_fonts:
        plt.rcParams["font.family"] = font_name
        print("사용하는 글꼴:", font_name)
        break
else:
    print("한글 글꼴을 찾지 못했습니다. 그래프의 한글이 깨지면 글꼴을 설치하세요.")

plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.figsize"] = (9, 5)


def decorate(ax, title, xlabel, ylabel, rotate=0):
    ax.set_title(title, pad=14)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.tick_params(axis="x", rotation=rotate)
    ax.grid(axis="y", alpha=0.25)
    plt.tight_layout()


def show_values(ax, fmt="{:,.0f}"):
    for container in ax.containers:
        ax.bar_label(container, fmt=fmt, padding=3)

### 3.1 분석 질문에 맞는 그래프 고르기

다음 여섯 가지 질문에 답하려면 어떤 그래프가 알맞은지 쓰세요. 그래프 이름과 **그렇게 고른 이유**를 한 줄로 쓰세요.

| 번호 | 알고 싶은 것 |
|:---|:---|
| ① | 계절별 평균 대여량의 크기 비교 |
| ② | 시간당 대여량이 어느 값 근처에 가장 많이 몰려 있는가 |
| ③ | 계절마다 대여량의 중앙값, 퍼진 정도, 이상값 후보의 차이 |
| ④ | 시간대가 바뀜에 따른 평균 대여량의 변화 |
| ⑤ | 기온이 높을수록 대여량이 많은가 |
| ⑥ | 계절을 같게 놓고 비 오는 시간과 비 안 오는 시간의 평균 비교 |

**답**

여기에 답을 쓰세요.

### 3.2 계절별 평균 대여량 막대그래프

계절별 평균 대여량을 막대그래프로 그리세요. 계절은 봄, 여름, 가을, 겨울 순서로 놓고, 제목과 축 이름을 붙이고 막대 위에 값을 표시하세요. 그래프에서 읽을 수 있는 내용을 두 문장으로 쓰세요.

**힌트** `.loc[["Spring", "Summer", "Autumn", "Winter"]]`로 순서를 정하고, `decorate()`, `show_values()`

### 3.3 대여량 분포 히스토그램

`operating`의 시간당 대여량 분포를 히스토그램으로 그리고, 평균과 중앙값을 구하세요. 분포의 모양은 어떻습니까? 평균과 중앙값은 각각 히스토그램의 어느 구간에 있으며, 어느 쪽이 더 큽니까? 그 차이는 분포의 모양과 어떤 관계가 있습니까?

### 3.4 계절별 대여량 상자그림

계절별 대여량 분포를 상자그림으로 비교하세요. 중앙값이 가장 높은 계절과 상자의 높이(IQR)가 가장 큰 계절은 어디입니까? (상자는 계절 이름의 알파벳 순서로 놓입니다.) 상자 위로 이어진 점들은 무엇이며, 1.16에서 확인한 이상값 후보와 같은 종류입니까? 이 점들을 **오류로 보고 삭제해도 되는지** 판단하세요.

**힌트** `boxplot(column="rented_count", by="season")`, 자동 제목은 `plt.suptitle("")`로 지웁니다.

### 3.5 공휴일 여부별 시간대 평균 대여량 선그래프

공휴일과 공휴일이 아닌 때를 두 개의 선으로 나누어, 시간대(0~23시)별 평균 대여량을 선그래프로 그리세요. 범례와 제목, 축 이름을 붙이세요. 두 선의 **최고점 시간대**는 언제입니까? 공휴일이 아닌 때에만 나타나는 모양이 있습니까? 이 그래프를 해석할 때 **공휴일 기록의 수**를 어떻게 고려해야 합니까?

**힌트** `groupby(["hour", "holiday"])` → `mean()` → `unstack()` → `plot(kind="line")`

### 3.6 기온과 대여량의 산점도

기온(`temperature`)과 대여량의 관계를 산점도로 그리세요. 점이 많이 겹치므로 점을 작고 투명하게 그리세요. 상관계수도 구하세요. 기온이 높을수록 대여량이 어떻게 달라집니까? 같은 기온에서도 대여량이 크게 퍼져 있는 이유가 될 만한 **다른 변수**를 하나 쓰고, 상관계수만으로 "기온 때문에 늘어난다"고 말할 수 없는 이유를 쓰세요.

**힌트** `plot(kind="scatter", x=…, y=…, s=…, alpha=…)`, `corr()`

**확인값** 상관계수 0.56

### 3.7 계절과 강수 여부별 묶은 막대그래프

1.10에서 만든 `is_rain` 열로 비 오는 시간과 비 안 오는 시간의 평균 대여량을 **계절을 같게 놓고** 비교하려고 합니다. 계절별로 두 평균이 막대 두 개로 나란히 놓이는 묶은 막대그래프를 그리세요. 모든 계절에서 비 오는 쪽이 낮습니까? 그래프에는 나타나지 않지만 해석할 때 반드시 함께 읽어야 하는 정보는 무엇입니까?

**힌트** `groupby(["season", "is_rain"])` → `mean()` → `unstack()` → `plot(kind="bar")`

**확인값** 여름 비 안 옴 1,110건, 비 옴 258건

### 3.8 축의 시작점 점검

3.2의 계절별 평균 막대그래프를 **y축이 `700`에서 시작하도록** 바꿔 다시 그리세요. 봄과 가을의 차이가 0에서 시작할 때와 비교해 어떻게 다르게 보입니까? 막대그래프의 y축은 어디에서 시작해야 하며 이유는 무엇입니까? 같은 규칙이 선그래프(3.5)에도 적용됩니까?

**힌트** `set_ylim(700, 1100)`

### 3.9 그래프를 문장으로 해석하기

3.2~3.7의 그래프 중 하나를 골라, 다음 세 부분으로 나누어 쓰세요.

1. **확인한 사실** 그래프에서 읽은 내용을 숫자와 함께 씁니다.
2. **가능한 해석** 왜 그럴 수 있는지 씁니다. 원인을 단정하지 않습니다.
3. **한계** 이 그래프로는 알 수 없는 것을 씁니다. 표본이 작은 그룹이나 통제하지 못한 조건을 포함합니다.

**답**

여기에 답을 쓰세요.

### 3.10 분석 질문에 답하는 보고용 그래프

처음 질문으로 돌아갑니다. "서울 공공자전거의 시간대별 평균 대여량은 어떻게 다르며, 어느 시간대에 대여가 가장 많은가?" 이 질문에 답하는 **보고용 그래프 한 장**을 만드세요.

- 그래프 종류를 고르고 이유를 쓰세요.
- 제목과 축 이름을 붙이세요.
- 가장 많은 시간대와 그 평균을 **코드로** 구하세요.
- 그래프 아래에 **결론을 한 문장**으로 쓰세요. 전처리 기준(1.15)과 자료의 범위(서울 전체 합계, 1년)도 밝히세요.